# Rol 5 — Día 4: meta-evaluación reproducible

Este notebook **lee y analiza** los CSV del factorial y baseline pareado del Día 3. No ejecuta el AG, el baseline ni la función de aptitud. Valida la matriz guardada, resume la comparación AG vs búsqueda aleatoria y estima efectos factoriales de población, elitismo y probabilidad de mutación.

## Alcance y procedencia

La ruta de datos está fijada a la ejecución `ejecucion_20261021T184758Z_c5dc431`. STFT y chroma se procesan por separado. Las cinco semillas y las doce configuraciones deben estar completas; cualquier mezcla, duplicado o presupuesto AG/baseline inconsistente detiene el análisis. No se hacen nuevas evaluaciones experimentales.

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True

RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'analizar_rol5_dia4.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('No se encontró la raíz del repositorio.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from scripts.analizar_rol5_dia4 import (
    METRICAS, RUTA_FACTORIAL, ajustar_holm, anova_factorial,
    cargar_y_validar, efectos_marginales, graficar_efectos, resumen_pareado,
)

datos = cargar_y_validar(RUTA_FACTORIAL)
print(f'Resultados cargados y validados desde: {RUTA_FACTORIAL.relative_to(RAIZ)}')
print('Se analizarán 60 corridas AG y 60 baselines por métrica; no se ejecutó el AG.')

In [ ]:
import json

resumenes_pareados = {}
for metrica in METRICAS:
    resumenes_pareados[metrica] = resumen_pareado(datos[metrica]['pareadas'])
    print(f'\n{metrica.upper()} — comparación pareada descriptiva')
    print(json.dumps(resumenes_pareados[metrica], indent=2, ensure_ascii=False))

In [ ]:
anovas = {metrica: anova_factorial(datos[metrica]['ag']) for metrica in METRICAS}
ajustar_holm(anovas)  # familia conjunta: 7 términos × 2 métricas
for metrica in METRICAS:
    print(f'\n{metrica.upper()} — ANOVA factorial completo y p de Holm')
    print(json.dumps(anovas[metrica], indent=2, ensure_ascii=False))

In [ ]:
efectos = {metrica: efectos_marginales(datos[metrica]['ag']) for metrica in METRICAS}
for metrica, contraste in efectos.items():
    print(f'{metrica.upper()} — contrastes marginales descriptivos')
    print(json.dumps(contraste, indent=2, ensure_ascii=False))

figura = graficar_efectos(datos)
figura

## Interpretación y límites

Los contrastes del gráfico son cambios marginales entre los niveles extremos observados; no son una sensibilidad de genes. El ANOVA incluye interacciones y corrige sus 14 pruebas con Holm. Estos análisis son exploratorios: cinco semillas por configuración, una melodía y una referencia; no hay evaluación auditiva ciega ni validación perceptual. La comparación AG–baseline se informa descriptivamente; que los pares observados favorezcan al AG no prueba superioridad general. La ratificación colectiva de STFT como métrica primaria y de la configuración de operadores B continúa pendiente.